Este proyecto de análisis de datos explora la relación entre el mercado financiero y el mercado de hardware de consumo (2022-2024). En lugar de asumir causalidad directa (la creencia popular de que Wall Street "dicta" los precios en las tiendas), investigaremos si las acciones bursátiles funcionan como indicadores adelantados (leading indicators).

Analizaremos empíricamente si factores macroeconómicos subyacentes (como la disrupción de la Inteligencia Artificial o el colapso de la minería cripto) generan un acoplamiento temporal entre la bolsa y los precios retail, o si el mercado de GPUs opera de forma totalmente desacoplada, regido por sus propias leyes de oferta y demanda en el mercado secundario.

Es muy probable que, a medida que profundicemos en las correlaciones y agrupaciones, los mismos datos nos revelen nuevos interrogantes y desvíos interesantes que enriquezcan esta investigación. Finalmente, la culminación de este proyecto consistirá en exportar nuestros hallazgos para construir un dashboard interactivo en Tableau, transformando todo este procesamiento analítico en un storytelling visual claro y de alto impacto.

In [1]:
import pandas as pd
from scipy.stats import pearsonr

# 1. Cargamos los datasets limpios generados en la fase de Data Cleaning
df_stocks = pd.read_csv('cleaned_market_stocks_monthly.csv')
df_gpus = pd.read_csv('cleaned_gpu_prices_monthly.csv')

# 2. Nos aseguramos de que la columna de fecha sea de tipo datetime para el cruce
df_stocks['month_date'] = pd.to_datetime(df_stocks['month_date'])
df_gpus['month_date'] = pd.to_datetime(df_gpus['month_date'])

print("Datasets cargados correctamente.")

Datasets cargados correctamente.


### 2. Preparación de los datos (Agrupación por marca y Merge)
Para que el análisis sea preciso, no podemos mezclar las acciones de NVIDIA con los precios de las placas AMD, ni viceversa. 
Primero, vamos a separar las GPUs en dos DataFrames distintos buscando las palabras clave "GeForce" y "Radeon". Luego, calcularemos el promedio general del precio de las GPUs por marca para cada mes y lo uniremos con el valor de las acciones correspondiente.

In [2]:
# Filtramos y promediamos los precios de las GPUs por marca para cada mes
df_nvidia_gpus = df_gpus[df_gpus['name'].str.contains('GeForce', na=False)].groupby('month_date')[['avg_retail_price', 'avg_used_price']].mean().reset_index()
df_amd_gpus = df_gpus[df_gpus['name'].str.contains('Radeon', na=False)].groupby('month_date')[['avg_retail_price', 'avg_used_price']].mean().reset_index()

# Hacemos el cruce (MERGE) con las tablas de acciones
# Usamos inner join para quedarnos solo con los meses donde tenemos ambos datos
df_nvda_full = pd.merge(df_nvidia_gpus, df_stocks[['month_date', 'avg_nvda_close']], on='month_date', how='inner').dropna()
df_amd_full = pd.merge(df_amd_gpus, df_stocks[['month_date', 'avg_amd_close']], on='month_date', how='inner').dropna()

df_nvda_full.head()

,month_date,avg_retail_price,avg_used_price,avg_nvda_close
0,2022-11-01,555.071429,368.357143,15.30
1,2022-12-01,763.400000,459.800000,16.20
2,2023-01-01,735.062500,430.333333,17.26
3,2023-02-01,654.625000,425.625000,22.02
4,2023-03-01,572.235294,376.388889,25.09


### 3. Prueba de Hipótesis: Nivel de Significancia vs P-value
**Hipótesis a evaluar:** El valor en bolsa de la empresa afecta los precios de sus GPUs nuevas y usadas.
* **Hipótesis Nula ($H_0$):** No existe correlación lineal entre el valor de la acción y el precio de las GPUs ($r = 0$).
* **Hipótesis Alternativa ($H_1$):** Existe una correlación significativa ($r \neq 0$).
* **Nivel de Significancia ($\alpha$):** 0.05.

A continuación, crearemos una función que compare el $p$-value obtenido contra nuestro $\alpha$ e imprima el veredicto estadístico.

In [3]:
from scipy.stats import pearsonr

def evaluar_correlacion(df, col_x, col_y, nombre_prueba, lag=0):
    """
    Calcula la correlación de Pearson aplicando un desfase (lag) opcional.
    """
    # Creamos un dataframe temporal solo con las dos columnas para no alterar el original
    df_temp = df[[col_x, col_y]].copy()
    
    # Si pasamos un lag > 0, desplazamos la variable de precios (col_y) hacia atrás
    if lag > 0:
        df_temp[col_y] = df_temp[col_y].shift(-lag)
        
    # Eliminamos los nulos generados por el lag
    df_temp = df_temp.dropna()
    
    # Calculamos Pearson r y p-value
    r, p_value = pearsonr(df_temp[col_x], df_temp[col_y])
    alpha = 0.05
    
    print(f"--- Prueba: {nombre_prueba} (Lag {lag}M) ---")
    print(f"Coeficiente de Correlación (r): {r:.4f}")
    print(f"P-value: {p_value:.6f}")
    
    if p_value < alpha:
        print("Veredicto: p-value < 0.05. Rechazamos H0.")
        print("Conclusión: Existe una correlación ESTADÍSTICAMENTE SIGNIFICATIVA.\n")
    else:
        print("Veredicto: p-value >= 0.05. No podemos rechazar H0.")
        print("Conclusión: NO hay evidencia suficiente para afirmar correlación.\n")

In [4]:
# Ejecutamos las pruebas para NVIDIA
evaluar_correlacion(df_nvda_full, 'avg_nvda_close', 'avg_retail_price', "NVIDIA Stock vs GeForce NUEVAS", 0)
evaluar_correlacion(df_nvda_full, 'avg_nvda_close', 'avg_used_price', "NVIDIA Stock vs GeForce USADAS", 0)

# Ejecutamos las pruebas para AMD
evaluar_correlacion(df_amd_full, 'avg_amd_close', 'avg_retail_price', "AMD Stock vs Radeon NUEVAS", 0)
evaluar_correlacion(df_amd_full, 'avg_amd_close', 'avg_used_price', "AMD Stock vs Radeon USADAS", 0)

--- Prueba: NVIDIA Stock vs GeForce NUEVAS (Lag 0M) ---
Coeficiente de Correlación (r): 0.0744
P-value: 0.723636
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: NVIDIA Stock vs GeForce USADAS (Lag 0M) ---
Coeficiente de Correlación (r): -0.1193
P-value: 0.570034
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: AMD Stock vs Radeon NUEVAS (Lag 0M) ---
Coeficiente de Correlación (r): -0.2101
P-value: 0.313381
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: AMD Stock vs Radeon USADAS (Lag 0M) ---
Coeficiente de Correlación (r): -0.3702
P-value: 0.068532
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.



### 📊 Conclusiones del Análisis en Tiempo Real (Lag 0)

Tras ejecutar la prueba de correlación de Pearson comparando los precios de las acciones y el valor de las GPUs en el mismo mes exacto, **no se encontró evidencia estadísticamente significativa (p > 0.05)** en ninguno de los escenarios planteados.

Esto nos permite rechazar la intuición común de que el mercado de consumo reacciona de forma inmediata a los movimientos bursátiles. "Wall Street" y el "Retail" operan a velocidades distintas. El valor de empresas como NVIDIA o AMD puede dispararse hoy por anuncios corporativos (como nuevos chips para servidores de IA), pero ese impacto no se traduce mágicamente en un aumento o bajada de precios en las tiendas de hardware de un día para el otro. Para encontrar una relación real, debemos incorporar a nuestro modelo analítico el retraso natural de la cadena de suministro y la inercia del mercado.

In [5]:
# Ejecutamos las pruebas para NVIDIA con lag de 2 meses
evaluar_correlacion(df_nvda_full, 'avg_nvda_close', 'avg_retail_price', "NVIDIA Stock vs GeForce NUEVAS (Lag 2M)", 2)
evaluar_correlacion(df_nvda_full, 'avg_nvda_close', 'avg_used_price', "NVIDIA Stock vs GeForce USADAS (Lag 2M)", 2)

# Ejecutamos las pruebas para AMD con lag de 2 meses
evaluar_correlacion(df_amd_full, 'avg_amd_close', 'avg_retail_price', "AMD Stock vs Radeon NUEVAS (Lag 2M)", 2)
evaluar_correlacion(df_amd_full, 'avg_amd_close', 'avg_used_price', "AMD Stock vs Radeon USADAS (Lag 2M)", 2)

--- Prueba: NVIDIA Stock vs GeForce NUEVAS (Lag 2M) (Lag 2M) ---
Coeficiente de Correlación (r): 0.2045
P-value: 0.349172
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: NVIDIA Stock vs GeForce USADAS (Lag 2M) (Lag 2M) ---
Coeficiente de Correlación (r): -0.0733
P-value: 0.739610
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: AMD Stock vs Radeon NUEVAS (Lag 2M) (Lag 2M) ---
Coeficiente de Correlación (r): -0.3332
P-value: 0.120278
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: AMD Stock vs Radeon USADAS (Lag 2M) (Lag 2M) ---
Coeficiente de Correlación (r): -0.4890
P-value: 0.017891
Veredicto: p-value < 0.05. Rechazamos H0.
Conclusión: Existe una correlación ESTADÍSTICAMENTE SIGNIFICATIVA.



In [6]:
# Ejecutamos las pruebas para NVIDIA con lag de 3 meses
evaluar_correlacion(df_nvda_full, 'avg_nvda_close', 'avg_retail_price', "NVIDIA Stock vs GeForce NUEVAS (Lag 3M)", 3)
evaluar_correlacion(df_nvda_full, 'avg_nvda_close', 'avg_used_price', "NVIDIA Stock vs GeForce USADAS (Lag 3M)", 3)

# Ejecutamos las pruebas para AMD con lag de 3 meses
evaluar_correlacion(df_amd_full, 'avg_amd_close', 'avg_retail_price', "AMD Stock vs Radeon NUEVAS (Lag 3M)", 3)
evaluar_correlacion(df_amd_full, 'avg_amd_close', 'avg_used_price', "AMD Stock vs Radeon USADAS (Lag 3M)", 3)

--- Prueba: NVIDIA Stock vs GeForce NUEVAS (Lag 3M) (Lag 3M) ---
Coeficiente de Correlación (r): 0.4432
P-value: 0.038840
Veredicto: p-value < 0.05. Rechazamos H0.
Conclusión: Existe una correlación ESTADÍSTICAMENTE SIGNIFICATIVA.

--- Prueba: NVIDIA Stock vs GeForce USADAS (Lag 3M) (Lag 3M) ---
Coeficiente de Correlación (r): 0.0407
P-value: 0.857273
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: AMD Stock vs Radeon NUEVAS (Lag 3M) (Lag 3M) ---
Coeficiente de Correlación (r): -0.2354
P-value: 0.291716
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: AMD Stock vs Radeon USADAS (Lag 3M) (Lag 3M) ---
Coeficiente de Correlación (r): -0.4154
P-value: 0.054518
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.



### ⏱️ Conclusiones del Análisis de Desfase Temporal (Cross-Correlation)

Al introducir retrasos de 2 y 3 meses (Lag 2M y Lag 3M) para simular los tiempos de logística, stock y reacción del mercado, la estadística nos reveló dos comportamientos totalmente distintos y altamente significativos:

#### NVIDIA NUEVAS y el Acoplamiento Macro (Lag 3M, $p < 0.05$):
La valoración en bolsa funciona como un indicador adelantado de 3 meses para el precio en tiendas. No hay causalidad directa entre ambos, sino que se mueven en tándem impulsados por factores macroeconómicos comunes (como el shock de demanda corporativa por la IA y los tiempos de inercia logística).

#### AMD USADAS y el Desacople Financiero (Lag 2M, $p < 0.05$):
Los precios de AMD muestran un desacople total de su rendimiento accionario. La correlación negativa retardada frente al ecosistema cripto sugiere que este segmento se rige estrictamente por la oferta del consumidor: el mercado tardó ~60 días en absorber el exceso de inventario post-minería, lo que hundió los precios por saturación de oferta.

### 4. Impacto del Criptomercado: Ethereum vs GPUs
El evento "The Merge" (Septiembre 2022) eliminó la minería con tarjetas gráficas para Ethereum. Queremos comprobar si el valor de ETH siguió afectando los precios de las GPUs (por inercia del mercado o liquidación de granjas de minería) durante el periodo 2022-2024.

Primero, cargaremos el dataset de Ethereum, lo agruparemos por mes para que coincida con nuestra granularidad, y haremos el cruce (*merge*) con los precios de NVIDIA y AMD.

In [7]:
# 1. Cargamos el dataset de Ethereum
df_eth = pd.read_csv('ETH-USD.csv')

# 2. Limpieza de datos: Convertimos 'Close' a número real (Float)
# Pasamos a string para poder borrar comas de miles, y to_numeric convierte a float.
# errors='coerce' transforma cualquier texto basura (como "null") en NaN, el cual mean() ignora automáticamente.
df_eth['Close'] = pd.to_numeric(df_eth['Close'].astype(str).str.replace(',', ''), errors='coerce')

# 3. Limpieza de Fechas
df_eth['Date'] = pd.to_datetime(df_eth['Date'])
df_eth['month_date'] = df_eth['Date'].dt.to_period('M').dt.to_timestamp()

# 4. Agrupación Mensual
df_eth_monthly = df_eth.groupby('month_date')['Close'].mean().reset_index()
df_eth_monthly.rename(columns={'Close': 'avg_eth_close'}, inplace=True)

# 5. Cruzamos (MERGE) la data de Ethereum con los promedios de GPUs
df_nvda_eth = pd.merge(df_nvidia_gpus, df_eth_monthly, on='month_date', how='inner').dropna()
df_amd_eth = pd.merge(df_amd_gpus, df_eth_monthly, on='month_date', how='inner').dropna()

print("Dataset de Ethereum limpiado, agrupado y cruzado con éxito.")

Dataset de Ethereum limpiado, agrupado y cruzado con éxito.


### 5. Prueba de Correlación: ETH vs GPUs (Tiempo Real)
Evaluaremos si existe una relación directa en el mismo mes entre el precio promedio de Ethereum y los precios en las tiendas de componentes.

In [8]:
print("--- PRUEBA ETHEREUM: TIEMPO REAL (LAG 0) ---\n")

# NVIDIA vs ETH
evaluar_correlacion(df_nvda_eth, 'avg_eth_close', 'avg_retail_price', "Ethereum vs GeForce NUEVAS (Lag 0)", 0)
evaluar_correlacion(df_nvda_eth, 'avg_eth_close', 'avg_used_price', "Ethereum vs GeForce USADAS (Lag 0)", 0)

# AMD vs ETH
evaluar_correlacion(df_amd_eth, 'avg_eth_close', 'avg_retail_price', "Ethereum vs Radeon NUEVAS (Lag 0)", 0)
evaluar_correlacion(df_amd_eth, 'avg_eth_close', 'avg_used_price', "Ethereum vs Radeon USADAS (Lag 0)", 0)

--- PRUEBA ETHEREUM: TIEMPO REAL (LAG 0) ---

--- Prueba: Ethereum vs GeForce NUEVAS (Lag 0) (Lag 0M) ---
Coeficiente de Correlación (r): 0.0489
P-value: 0.816326
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: Ethereum vs GeForce USADAS (Lag 0) (Lag 0M) ---
Coeficiente de Correlación (r): 0.0629
P-value: 0.765205
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: Ethereum vs Radeon NUEVAS (Lag 0) (Lag 0M) ---
Coeficiente de Correlación (r): -0.1845
P-value: 0.377204
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: Ethereum vs Radeon USADAS (Lag 0) (Lag 0M) ---
Coeficiente de Correlación (r): -0.3465
P-value: 0.089777
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.



### 6. Prueba de Correlación: ETH vs GPUs con Desfase Temporal (Lag 2 Meses)
Tal como descubrimos con la bolsa de valores, la logística y la saturación del mercado usado pueden tener un retraso. Aplicaremos un desfase de 2 meses para ver si las caídas de Ethereum tardaron 60 días en impactar (o inundar) el mercado secundario de gráficas.

In [9]:
# --- ESCENARIO 2: DESFASE DE 2 MESES (LAG 2) ---
evaluar_correlacion(df_nvda_eth, 'avg_eth_close', 'avg_retail_price', 'ETH vs GeForce NUEVAS', lag=2)
evaluar_correlacion(df_nvda_eth, 'avg_eth_close', 'avg_used_price', 'ETH vs GeForce USADAS', lag=2)
evaluar_correlacion(df_amd_eth, 'avg_eth_close', 'avg_retail_price', 'ETH vs Radeon NUEVAS', lag=2)
evaluar_correlacion(df_amd_eth, 'avg_eth_close', 'avg_used_price', 'ETH vs Radeon USADAS', lag=2)

--- Prueba: ETH vs GeForce NUEVAS (Lag 2M) ---
Coeficiente de Correlación (r): 0.1011
P-value: 0.646256
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: ETH vs GeForce USADAS (Lag 2M) ---
Coeficiente de Correlación (r): -0.0571
P-value: 0.795841
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: ETH vs Radeon NUEVAS (Lag 2M) ---
Coeficiente de Correlación (r): -0.3510
P-value: 0.100585
Veredicto: p-value >= 0.05. No podemos rechazar H0.
Conclusión: NO hay evidencia suficiente para afirmar correlación.

--- Prueba: ETH vs Radeon USADAS (Lag 2M) ---
Coeficiente de Correlación (r): -0.5032
P-value: 0.014388
Veredicto: p-value < 0.05. Rechazamos H0.
Conclusión: Existe una correlación ESTADÍSTICAMENTE SIGNIFICATIVA.



### 7. Análisis de Volatilidad y Riesgo: El Coeficiente de Variación (CV)

En los mercados financieros, no basta con saber si el precio subió o bajó; es fundamental entender la **volatilidad** (qué tanto fluctúa el precio alrededor de su promedio). Para este análisis, utilizaremos el **Coeficiente de Variación**, que se calcula como la desviación estándar dividida por la media ($CV = \frac{\sigma}{\mu}$).

Al multiplicar este resultado por 100, obtenemos un porcentaje. Podremos ver si el mercado de GPUs usadas (un activo físico) fue porcentualmente más inestable y riesgoso que invertir directamente en Ethereum o en la bolsa de valores durante el período 2022-2024.

In [10]:
def calcular_cv(serie, nombre_activo):
    """
    Calcula el Coeficiente de Variación (CV) en porcentaje para una serie de datos de pandas.
    """
    # Calculamos la media y la desviación estándar
    media = serie.mean()
    desvio = serie.std()
    
    # Calculamos el CV y lo pasamos a porcentaje
    cv = (desvio / media) * 100
    
    return {'Activo / Mercado': nombre_activo, 'Volatilidad (CV %)': round(cv, 2)}

In [11]:
# Recopilamos las volatilidades de todos nuestros activos en una lista
resultados_volatilidad = [
    # Acciones y Cripto
    calcular_cv(df_stocks['avg_nvda_close'], '1. Stock Market: NVIDIA (NVDA)'),
    calcular_cv(df_stocks['avg_amd_close'], '1. Stock Market: AMD (AMD)'),
    calcular_cv(df_eth_monthly['avg_eth_close'], '1. Cryptocurrency: Ethereum (ETH)'),
    
    # Mercado de GPUs NVIDIA
    calcular_cv(df_nvidia_gpus['avg_retail_price'], '2. Retail Market: NEW GeForce'),
    calcular_cv(df_nvidia_gpus['avg_used_price'], '3. Secondary Market: USED GeForce'),
    
    # Mercado de GPUs AMD
    calcular_cv(df_amd_gpus['avg_retail_price'], '2. Retail Market: NEW Radeon'),
    calcular_cv(df_amd_gpus['avg_used_price'], '3. Secondary Market: USED Radeon')
]

# Convertimos a DataFrame para visualizarlo de forma ordenada
df_volatilidad = pd.DataFrame(resultados_volatilidad)

# Ordenamos de mayor a menor volatilidad para ver qué mercado fue el más salvaje
df_volatilidad = df_volatilidad.sort_values(by='Volatilidad (CV %)', ascending=False).reset_index(drop=True)

print("--- RANKING DE VOLATILIDAD (2022-2024) ---")
display(df_volatilidad)

--- RANKING DE VOLATILIDAD (2022-2024) ---


,Activo / Mercado,Volatilidad (CV %)
0,1. Cryptocurrency: Ethereum (ETH),82.00
1,1. Stock Market: NVIDIA (NVDA),63.60
2,1. Stock Market: AMD (AMD),28.81
3,3. Secondary Market: USED Radeon,18.34
4,2. Retail Market: NEW Radeon,11.15
5,2. Retail Market: NEW GeForce,10.22
6,3. Secondary Market: USED GeForce,8.23


In [12]:
# Reemplazá 'df_volatilidad' por el nombre real de tu variable
df_volatilidad.to_csv('volatility_ranking.csv', index=False)

print("¡Tabla de riesgo exportada con éxito!")

¡Tabla de riesgo exportada con éxito!


### 💡 Interpretación de la Volatilidad (Conclusiones 2022-2024)

Al observar el ranking de volatilidad (CV %), los datos revelan una radiografía perfecta de cómo se estructuró el riesgo y la inestabilidad en este ecosistema:

* **El mundo financiero (Cripto y Wall Street):** Como era de esperar, Ethereum lidera el ranking con una volatilidad extrema del 82.00%. Sin embargo, lo más destacable es la acción de NVIDIA (63.60%), que superó ampliamente a AMD (28.81%). Esto refleja el crecimiento explosivo y parabólico de NVDA impulsado por la Inteligencia Artificial, comportándose casi con la misma volatilidad que un activo cripto.
* **El ancla del mercado Retail:** Tal como dicta la teoría económica, los mercados *Retail* (nuevos) funcionaron como el ancla del ecosistema. Tanto Radeon Nuevas (11.15%) como GeForce Nuevas (10.22%) mantuvieron una volatilidad muy baja, demostrando que los precios sugeridos por los fabricantes (MSRP) y la cadena de distribución tradicional aíslan al consumidor de los picos financieros.
* **La gran divergencia del mercado de Usados:** Aquí es donde la estadística brilla y confirma nuestras pruebas de correlación previas:
    * **Radeon Usadas (18.34%):** Es notablemente más volátil que su contraparte nueva. Esto cuantifica el "efecto resaca" de la minería de Ethereum. La inundación repentina de gráficas AMD de segunda mano generó inestabilidad y fuertes fluctuaciones de precio.
    * **GeForce Usadas (8.23%):** ¡Es el activo menos volátil de toda la tabla! A pesar de que la empresa matriz (NVDA) experimentaba una volatilidad financiera del 63%, su mercado secundario de gráficas se mantuvo como una roca. Esto sugiere una retención de valor altísima y una demanda *gamer* constante que absorbió cualquier oferta sin alterar los precios.


# Conclusion

### 💡 Interpretación de la Volatilidad (Conclusiones 2022-2024)

Al observar el ranking de volatilidad (CV %), los datos revelan una radiografía perfecta de cómo se estructuró el riesgo y la inestabilidad en este ecosistema:

* **El mundo financiero (Cripto y Wall Street):** Como era de esperar, Ethereum lidera el ranking con una volatilidad extrema del 82.00%. Sin embargo, lo más destacable es la acción de NVIDIA (63.60%), que superó ampliamente a AMD (28.81%). Esto refleja el crecimiento explosivo y parabólico de NVDA impulsado por la Inteligencia Artificial, comportándose casi con la misma volatilidad que un activo cripto.
* **El ancla del mercado Retail:** Tal como dicta la teoría económica, los mercados *Retail* (nuevos) funcionaron como el ancla del ecosistema. Tanto Radeon Nuevas (11.15%) como GeForce Nuevas (10.22%) mantuvieron una volatilidad muy baja, demostrando que los precios sugeridos por los fabricantes (MSRP) y la cadena de distribución tradicional aíslan al consumidor de los picos financieros.
* **La Brecha de Volatilidad en el Mercado de Usados:** Mientras los activos especulativos sufrieron fluctuaciones extremas, el mercado físico se dividió en dos realidades. El mercado secundario de **NVIDIA** funcionó como un ancla de estabilidad (casi 10 veces menos volátil que ETH). Por el contrario, **AMD Usadas** absorbió todo el impacto del colapso cripto, demostrando que no todo el hardware retiene su valor por igual ante el estrés macroeconómico.

---

### 🎯 Conclusión Final

**Veredicto:**
El mercado de GPUs no es un espejo en tiempo real de Wall Street, sino que opera mediante rezagos estructurales y dinámicas divergentes:

1. **NVIDIA (Inercia corporativa e IA):** Su valoración en bolsa es un **indicador adelantado de 3 meses** para los precios *retail*, moviéndose en conjunto por el impulso de la IA y la inercia de la cadena de suministro.
2. **AMD (Desacople y "Efecto Resaca"):** Está completamente desacoplada de la bolsa. Su cotización depende de la oferta y demanda del consumidor final, sufriendo un colapso con **2 meses de retraso** producto de la liquidación de inventario post-minería.

**Próximo paso:** Importar los datasets limpios a **Tableau** para visualizar estos *lags* temporales como indicadores de mercado, contrastar las brechas de riesgo mediante gráficos de dispersión y contextualizar los datos con hitos macroeconómicos clave. Acá abajo dejo el enlace al proyecto e imágenes de los dashboards.

https://public.tableau.com/app/profile/joaquin.carnota/viz/MainStreetvs_WallStreetTheGPUMarketDisconnect2022-2024/Title

<img src="assets/1-Title.png" width="1000">
<img src="assets/2-Dashboard.png" width="1000">
<img src="assets/3-Dashboard.png" width="1000">
<img src="assets/4-Dashboard.png" width="1000">